In [1]:
# Building the Road Network Graph from Lagos GeoPackage

#This notebooks loads road and POI data,  builds a NetworkX graph from roads, snaps POIs to nearest nodes, and saves outputs.

In [2]:
import os
import geopandas as gpd
import fiona
import networkx as nx
from shapely.geometry import LineString, MultiLineString
import numpy as np
from scipy.spatial import cKDTree
import pickle

In [3]:
# Find project root (go up two levels from the notebook dir)
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
print ("Project root:", project_root)

#Build full path to GeoPackage
data_path = os.path.join (project_root, "1_dataset", "raw", "processed", "lagos_data.gpkg")
print ("GPKG path:", data_path)

# List layers to verify
print ("Layers:", fiona.listlayers(data_path))

Project root: /home/ridwan-ayinde/Desktop/MIT Emerging Talent/Final Project
GPKG path: /home/ridwan-ayinde/Desktop/MIT Emerging Talent/Final Project/1_dataset/raw/processed/lagos_data.gpkg
Layers: ['lagos_boundaryshp__nga_admbnda_adm2_osgof_20170222', 'roads_lagos', 'buildings_lagos', 'pois_lagos', 'buildings_clean', 'roads_clean', 'pois_clean']


In [4]:
# --- BUILD ROAD NETWORK GRAPH ---

print ("Loading roads from Lagos GeoPackage...")
roads_clean_final = gpd.read_file(data_path, layer = "roads_clean")
print (roads_clean_final.head())

#Reproject to metric CRS (Web Mercator EPSG: 3857 for Lagos)
roads_clean_final_m = roads_clean_final.to_crs(epsg=3857)
print (f"Reprojected CRS:", roads_clean_final_m.crs)

Loading roads from Lagos GeoPackage...
                                            geometry
0  MULTILINESTRING ((3.4039 6.45847, 3.40274 6.45...
1  MULTILINESTRING ((3.40403 6.44011, 3.40472 6.4...
2  MULTILINESTRING ((3.38844 6.46564, 3.38792 6.4...
3  MULTILINESTRING ((3.39968 6.45921, 3.39912 6.4...
4  MULTILINESTRING ((3.408 6.45503, 3.40795 6.454...
Reprojected CRS: EPSG:3857


In [5]:
def roads_to_edges(gdf, precision=3):
    """
    Convert road geometries to graph edges with lengths.
    Handles LineString and MultiLineString; rounds coords for presion

    """
    
    edges = []
    for idx, row in gdf.iterrows():
        geom = row.geometry
        if geom is None:
            continue
        if isinstance (geom, LineString):
            coords = list(geom.coords)
        elif isinstance (geom, MultiLineString):
            coords = []
            for line in geom.geoms:
                if isinstance (line, LineString):
                    coords += list(line.coords)
                    
                # Skip if sub-geom is not LineSting
        else:
            print(f"Skipping unsupported geometry type for row {idx}: {type(geom)}")
            continue
        
        # Round for floating-point stability
        coords = [(round(x, precision), round(y, precision)) for x, y in coords]
        
        # Create edges between consecutive points
        for i in range (len(coords)-1):
            u = coords[i]
            v = coords[i + 1]
            length = LineString ([u,v]).length # Distance in projected CRS units (meters)
            edges.append((u,v, {"length": length}))
    return edges

In [6]:
# Generate Edges

edges = roads_to_edges(roads_clean_final_m)
print (f"Generated {len(edges)} edges")

#Build undirected graph

G = nx.Graph()
G.add_edges_from(edges)
print ("Graph built successfully")
print (f"Number of nodes: {G.number_of_nodes()}")
print (f"Number of edges: {G.number_of_edges()}")

Generated 16310 edges
Graph built successfully
Number of nodes: 15248
Number of edges: 16310


In [7]:
# ___ LOAD AND SNAP POIS ---

print ("Loading POIs...")
pois_clean = gpd.read_file(data_path, layer ='pois_clean')
print ("Loaded POIs:")
print(pois_clean.head())
print ("CRS:", pois_clean.crs)

#Reproject POIs to match graph CRS
pois_clean_m = pois_clean.to_crs (roads_clean_final_m.crs)

# Prepare POI centroids

pois_points = pois_clean_m.copy()
pois_points["geometry"] = pois_points.geometry.centroid
print(pois_points.head())
print ("Sample centroid:", pois_points.geometry.iloc[0])

Loading POIs...


Loaded POIs:
         fclass                                           geometry
0  market_place  MULTIPOLYGON (((3.38331 6.4626, 3.38336 6.4626...
1  market_place  MULTIPOLYGON (((3.3912 6.46051, 3.39146 6.4606...
2          park  MULTIPOLYGON (((3.39585 6.44864, 3.39638 6.449...
3   arts_centre  MULTIPOLYGON (((3.39428 6.44899, 3.39446 6.449...
4         pitch  MULTIPOLYGON (((3.39449 6.45098, 3.39466 6.451...
CRS: EPSG:4326
         fclass                       geometry
0  market_place  POINT (376779.993 720951.273)
1  market_place  POINT (377732.671 720748.183)
2          park  POINT (378102.319 719409.507)
3   arts_centre  POINT (377880.409 719423.976)
4         pitch  POINT (377906.133 719672.125)
Sample centroid: POINT (376779.99275387503 720951.2728126525)


In [8]:
# Debug: Check available columns
print("Available columns in pois_points:")
print(pois_points.columns.tolist())
print("\nFirst few rows:")
print(pois_points.head())

Available columns in pois_points:
['fclass', 'geometry']

First few rows:
         fclass                       geometry
0  market_place  POINT (376779.993 720951.273)
1  market_place  POINT (377732.671 720748.183)
2          park  POINT (378102.319 719409.507)
3   arts_centre  POINT (377880.409 719423.976)
4         pitch  POINT (377906.133 719672.125)


In [9]:
# Build KDTree for snapping POIs to nearest graph nodes

nodes_array = np.array (list(G.nodes)) # Convert to array for KDTree
tree = cKDTree (nodes_array)

# Get POI coordinates
poi_coords = np.array([[geom.x, geom.y]for geom in pois_points.geometry])

# Find nearest nodes

distances, indices = tree.query(poi_coords, k=1)
pois_points["nearest_node"] = [tuple(nodes_array[i]) for i in indices]
pois_points["snap_distance"] = distances

#Verify a few
print (pois_points[["fclass", "geometry", "nearest_node", "snap_distance"]].head())

         fclass                       geometry              nearest_node  \
0  market_place  POINT (376779.993 720951.273)  (376842.915, 720919.121)   
1  market_place  POINT (377732.671 720748.183)  (377726.413, 720729.665)   
2          park  POINT (378102.319 719409.507)  (378082.368, 719409.216)   
3   arts_centre  POINT (377880.409 719423.976)  (377866.976, 719466.665)   
4         pitch  POINT (377906.133 719672.125)  (377946.581, 719658.794)   

   snap_distance  
0      70.660796  
1      19.546977  
2      19.953265  
3      44.752749  
4      42.588509  


In [10]:
# ---SAVE OUTPUTS ---

os.makedirs("data", exist_ok = True)

# Save graph

with open ("data/road_graph.pkl", "wb") as f:
    pickle.dump(G, f)
print ("Saved graph to data/road_graph.pkl")

# Save POIs with snapped nodes

# Instead of GeoJSON, save as Parquet (keeps tuples intact)
pois_points.to_file("data/pois_with_nodes.geojson", driver="GeoJSON")
print("Saved POIs to data/pois_with_nodes.geojson")

Saved graph to data/road_graph.pkl


Saved POIs to data/pois_with_nodes.geojson
